In [1]:
import pandas as pd

In [2]:
# 1. Carga dos dados com o encoding correto (UTF-8)
salario = pd.read_csv('salario_prof.csv', sep=';', encoding='utf-8')

# 2. Normalização: Remove espaços invisíveis no início e fim dos nomes das colunas
# No seu arquivo a coluna está como " Salário inicial " (com espaços)
salario.columns = salario.columns.str.strip()

# 3. Limpeza e Conversão de Dados
salario['Salário inicial'] = (
    salario['Salário inicial']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 4. Encontrar o Estado com menor salário
# Limpamos a coluna UF também por segurança
salario['UF'] = salario['UF'].str.strip()
menor_salario = salario.loc[salario['Salário inicial'].idxmin()]

print("--- Análise de BI: Resultado Final ---")
print(f"Estado com menor salário: {menor_salario['UF']}")
print(f"Valor: R$ {menor_salario['Salário inicial']:,.2f}")
print(f"Região: {menor_salario['Região'].strip()}")


--- Análise de BI: Resultado Final ---
Estado com menor salário: MG
Valor: R$ 4,867.97
Região: Sudeste


In [3]:
# 5. Encontrar o Estado com maior salário
maior_salario = salario.loc[salario['Salário inicial'].idxmax()]

print("--- Análise de BI: Resultado Final ---")
print(f"Estado com menor salário: {menor_salario['UF']}")
print(f"Valor: R$ {menor_salario['Salário inicial']:,.2f}")
print(f"Região: {menor_salario['Região'].strip()}")


--- Análise de BI: Resultado Final ---
Estado com menor salário: MG
Valor: R$ 4,867.97
Região: Sudeste


In [4]:
# 1. Criar o ranking (ordenando do maior salário para o menor)
salario_ranking = salario.sort_values(by='Salário inicial', ascending=False).reset_index(drop=True)

# 2. Adicionar uma coluna de 'Posição' (Ranking) começando em 1
salario_ranking['Posição'] = salario_ranking.index + 1

# 3. Localizar a linha de Goiás (GO)
posicao_go = salario_ranking[salario_ranking['UF'] == 'GO']

print("--- Posicionamento de Mercado: GOIÁS ---")
print(posicao_go[['Posição', 'UF', 'Salário inicial', 'Região']])


--- Posicionamento de Mercado: GOIÁS ---
    Posição  UF  Salário inicial         Região
17       18  GO          5160.49   Centro-Oeste


In [5]:
# 1. Calcular a média aritmética de todos os estados
media_nacional = salario['Salário inicial'].mean()

# 2. Exibir o resultado formatado
print("--- Indicador de Desempenho (KPI) ---")
print(f"A média salarial nacional é: R$ {media_nacional:,.2f}")

# 3. Insight Comparativo: Quantos estados estão acima da média?
acima_da_media = salario[salario['Salário inicial'] > media_nacional].shape[0]
print(f"Estados acima da média: {acima_da_media}")
print(f"Estados abaixo da média: {len(salario) - acima_da_media}")


--- Indicador de Desempenho (KPI) ---
A média salarial nacional é: R$ 6,264.08
Estados acima da média: 9
Estados abaixo da média: 17


In [6]:
# 1. Agrupar por Região e calcular a média do Salário Inicial
# O strip() na Região garante que não tenhamos nomes duplicados por espaços
salario['Região'] = salario['Região'].str.strip()
media_por_regiao = salario.groupby('Região')['Salário inicial'].mean().sort_values(ascending=False)

# 2. Exibir o resultado formatado
print("--- Média Salarial por Região ---")
print(media_por_regiao.apply(lambda x: f"R$ {x:,.2f}"))


--- Média Salarial por Região ---
Região
Centro-Oeste    R$ 7,984.69
Norte           R$ 6,405.99
Nordeste        R$ 6,100.88
Sudeste         R$ 5,372.98
Sul             R$ 5,019.47
Name: Salário inicial, dtype: object


In [9]:
# 3. Análise: Encontrar a linha do maior salário para cada região
# O idxmax() encontra o índice do maior valor, e o loc recupera a linha inteira (UF, Valor, Região)
maiores_por_regiao = salario.loc[salario.groupby('Região')['Salário inicial'].idxmax()]

# 4. Ordenar do maior para o menor e imprimir (Print Único)
print("--- MAIOR SALÁRIO INICIAL POR REGIÃO ---")
print(maiores_por_regiao[['Região', 'UF', 'Salário inicial']].sort_values(by='Salário inicial', ascending=False).to_string(index=False))


--- MAIOR SALÁRIO INICIAL POR REGIÃO ---
       Região    UF  Salário inicial
 Centro-Oeste MS            13007.12
     Nordeste MA             8452.03
        Norte PA             8289.86
      Sudeste ES             5685.97
          Sul RS             5111.05


In [10]:
# 1. Encontrar o índice do menor salário inicial por região
idx_min_regiao = salario.groupby('Região')['Salário inicial'].idxmin()

# 2. Localizar as linhas e exibir em um print único (Região, UF, Valor)
print("--- MENOR SALÁRIO INICIAL POR REGIÃO ---")
print(salario.loc[idx_min_regiao, ['Região', 'UF', 'Salário inicial']].sort_values(by='Salário inicial').to_string(index=False))


--- MENOR SALÁRIO INICIAL POR REGIÃO ---
       Região    UF  Salário inicial
      Sudeste MG             4867.97
          Sul PR             4920.55
     Nordeste CE             4961.73
        Norte RO             5118.41
 Centro-Oeste GO             5160.49


In [13]:
# 1. Agrupar por região e calcular a média (usando o df que já está na memória)
media_regional = salario.groupby('Região')['Salário inicial'].mean().reset_index()

# 2. Exibir o resultado formatado em um print único
print("--- MÉDIA SALARIAL POR REGIÃO ---")
print(media_regional.sort_values(by='Salário inicial', ascending=False).to_string(index=False, float_format=lambda x: f"R$ {x:,.2f}"))

--- MÉDIA SALARIAL POR REGIÃO ---
       Região  Salário inicial
 Centro-Oeste      R$ 7,984.69
        Norte      R$ 6,405.99
     Nordeste      R$ 6,100.88
      Sudeste      R$ 5,372.98
          Sul      R$ 5,019.47


In [14]:
# 1. Calcular a média por região e identificar os extremos (maior e menor)
medias = salario.groupby('Região')['Salário inicial'].mean()

regiao_maior = medias.idxmax()
regiao_menor = medias.idxmin()

# 2. Exibir o resultado consolidado (Print Único)
print("--- ANÁLISE EXTREMA: MÉDIAS POR REGIÃO ---")
print(f"MAIOR MÉDIA: {regiao_maior} (R$ {medias[regiao_maior]:,.2f})")
print(f"MENOR MÉDIA: {regiao_menor} (R$ {medias[regiao_menor]:,.2f})")

--- ANÁLISE EXTREMA: MÉDIAS POR REGIÃO ---
MAIOR MÉDIA:  Centro-Oeste (R$ 7,984.69)
MENOR MÉDIA:  Sul (R$ 5,019.47)
